In [40]:
import numpy as np
import pandas as pd

from sksurv.metrics import concordance_index_censored
from cvxopt import matrix, solvers

def generate_wsvcr_data(n=500, censoring_rate=0.3, random_state=42):
    np.random.seed(random_state)

    x1 = np.random.normal(1, 1, n)
    x2 = np.random.normal(2, np.sqrt(2), n)
    x3 = np.random.exponential(1, n)

    x2_safe = np.where(np.abs(x2) < 1e-8, 1e-8, x2)

    lam = 5 * np.exp(
        -x1 * np.log(np.abs(x2_safe)) + np.sin(x3 ** 2)
    )

    k = 3
    U = np.random.uniform(0, 1, n)
    Y = lam * (-np.log1p(-U)) ** (1 / k)

    c_max = np.quantile(Y, 1 - censoring_rate)
    C = np.random.uniform(0, c_max, n)

    T = np.minimum(Y, C)
    delta = (Y <= C).astype(int)

    l = T.copy()
    u = np.where(delta == 1, T, np.inf)

    df = pd.DataFrame({
        "X1": x1,
        "X2": x2,
        "X3": x3,
        "Y_true": Y,
        "T_obs": T,
        "delta": delta,
        "l": l,
        "u": u
    })

    return df

In [49]:
import numpy as np
from cvxopt import matrix, solvers

solvers.options['show_progress'] = False

def wavelet_kernel(X, Y=None, A=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    return np.prod(
        np.cos(1.75 * diff / A) * np.exp(-0.5 * (diff / A) ** 2),
        axis=2
    )


def rbf_kernel(X, Y=None, gamma=1.0):
    if Y is None:
        Y = X
    diff = X[:, None, :] - Y[None, :, :]
    sq_dist = np.sum(diff**2, axis=2)
    return np.exp(-gamma * sq_dist)


def linear_kernel(X, Y=None):
    if Y is None:
        Y = X
    return X @ Y.T

class WSVCR_QP:
    def __init__(self, C=1.0, kernel="wavelet", kernel_params=None):
        self.C = C
        self.kernel = kernel
        self.kernel_params = kernel_params if kernel_params else {}

    def _compute_kernel(self, X, Y=None):
        if callable(self.kernel):
            return self.kernel(X, Y, **self.kernel_params)

        if self.kernel == "wavelet":
            return wavelet_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "rbf":
            return rbf_kernel(X, Y, **self.kernel_params)

        elif self.kernel == "linear":
            return linear_kernel(X, Y)

        else:
            raise ValueError("Unknown kernel")

    def fit(self, X, l, u):
        X = np.asarray(X)
        l = np.asarray(l)
        u = np.asarray(u)

        n = X.shape[0]

        L = np.isfinite(l)
        U = np.isfinite(u)

        # kernel matrix
        K = self._compute_kernel(X)

        # QP matrix
        P = np.block([
            [K, -K],
            [-K, K]
        ])

        P += 1e-8 * np.eye(2 * n)

        q = np.zeros(2 * n)

        for i in range(n):
            if L[i]:
                q[i] = -l[i]
            if U[i]:
                q[n + i] = u[i]

        A_eq = np.zeros((1, 2 * n))

        for i in range(n):
            if L[i]:
                A_eq[0, i] = 1
            if U[i]:
                A_eq[0, n + i] = -1

        b_eq = np.array([0.0])

        G = np.vstack([
            np.eye(2 * n),
            -np.eye(2 * n)
        ])

        h = np.hstack([
            self.C * np.ones(2 * n),
            np.zeros(2 * n)
        ])

        solution = solvers.qp(
            matrix(P),
            matrix(q),
            matrix(G),
            matrix(h),
            matrix(A_eq),
            matrix(b_eq)
        )

        z = np.array(solution['x']).flatten()

        self.alpha = z[:n]
        self.alpha_star = z[n:]
        self.theta = self.alpha - self.alpha_star

        self.X = X
        self.K = K

        self.b = self.compute_bias(l, u)

    def compute_bias(self, l, u):
        f = self.K @ self.theta

        b_vals = []

        for i in range(len(self.theta)):
            if self.alpha[i] > 1e-6 and np.isfinite(l[i]):
                b_vals.append(l[i] - f[i])
            elif self.alpha_star[i] > 1e-6 and np.isfinite(u[i]):
                b_vals.append(u[i] - f[i])

        return np.mean(b_vals) if b_vals else 0.0

    def predict(self, X_new):
        X_new = np.asarray(X_new)

        K_new = self._compute_kernel(self.X, X_new)

        return self.theta @ K_new + self.b


In [ ]:
df = generate_wsvcr_data(n=300, censoring_rate=0.1, random_state=42)

X = df[["X1", "X2", "X3"]].values
l = df["l"].values
u = df["u"].values

model = WSVCR_QP(C=2.0, kernel="wavelet", kernel_params={"A": 1})
model.fit(X, l, u)

y_pred = model.predict(X)

In [52]:
from sksurv.metrics import concordance_index_censored

def compute_cindex_sksurv(y_time, y_pred, delta):
    event = delta.astype(bool)

    risk = -y_pred

    cindex = concordance_index_censored(event, y_time, risk)[0]

    return cindex

In [56]:
from sklearn.model_selection import train_test_split

def train_test_eval(df, C, A, kernel):
    X = df[["X1","X2","X3"]].values
    l = df["l"].values
    u = df["u"].values
    y = df["T_obs"].values
    delta = df["delta"].values

    X_train, X_test, l_train, l_test, u_train, u_test, y_train, y_test, d_train, d_test = train_test_split(
        X, l, u, y, delta, test_size=0.3, random_state=None
    )

    model = WSVCR_QP(C=C, kernel=kernel, kernel_params={"A": A})
    model.fit(X_train, l_train, u_train)

    y_pred = model.predict(X_test)

    cindex = compute_cindex_sksurv(y_test, y_pred, d_test)

    return cindex

In [60]:
def grid_search(df):
    C_values = [0.25, 0.5, 1, 2, 4]
    A_values = [0.125, 0.25, 0.5, 1, 2, 4, 8]

    results = []

    for kernel in ["wavelet", "linear"]:
        for C in C_values:
            for A in A_values:
                scores = []

                for _ in range(100):
                    score = train_test_eval(df, C, A, kernel)
                    scores.append(score)

                results.append({
                    "kernel": kernel,
                    "C": C,
                    "A": A,
                    "mean_cindex": np.mean(scores),
                    "std": np.std(scores)
                })

                print(kernel, C, A, np.mean(scores))

    return pd.DataFrame(results)

In [ ]:
df = generate_wsvcr_data(n=300, censoring_rate=0.3)

results = grid_search(df)

print(results.sort_values("mean_cindex", ascending=False).head(10))

wavelet 0.25 0.125 0.47486515202976753
wavelet 0.25 0.25 0.5008116244175793
wavelet 0.25 0.5 0.5422184208954182
wavelet 0.25 1 0.6751934025132116
wavelet 0.25 2 0.7634136433074952
wavelet 0.25 4 0.7965662829286085
wavelet 0.25 8 0.8114974200267285
wavelet 0.5 0.125 0.4741207415389947
wavelet 0.5 0.25 0.5038787635645222
wavelet 0.5 0.5 0.5381230096875854
wavelet 0.5 1 0.6897839875447705
wavelet 0.5 2 0.786701065122145
wavelet 0.5 4 0.8183632032004655
wavelet 0.5 8 0.8109245391821227
wavelet 1 0.125 0.4875160663736717
wavelet 1 0.25 0.49279763905071583
wavelet 1 0.5 0.5436674408750366
wavelet 1 1 0.7047068316495227
wavelet 1 2 0.7853826054644256
wavelet 1 4 0.8106407309634082
wavelet 1 8 0.8155027613611651
wavelet 2 0.125 0.4866171491166017
wavelet 2 0.25 0.49644793781463364
wavelet 2 0.5 0.5481481072425697
wavelet 2 1 0.6932523522132338
wavelet 2 2 0.7924845488820722
wavelet 2 4 0.820634267498165
wavelet 2 8 0.8215474850754212
wavelet 4 0.125 0.48778227369109944
